# 2c. Catalog import and export: DCAT-AP, DCAT 3 and DSP, validated with SHACL

The catalog keeps one neutral RDF description of every resource and projects it per profile. This
notebook takes it both ways through the catalog agent's management API:

- **Out**: the Provider's catalog in DSP 2025-1, DCAT 3 and DCAT-AP 3.0.1, as JSON-LD, Turtle,
  N-Triples or RDF/XML. A preview keeps only the first datasets; a download has them all
  (`GET /api/v1/catalog-agent/catalogs/{id}/profiles/{profile}`).
- **In**: DCAT-AP or DCAT 3 documents validated with SHACL and loaded in one transaction
  (`POST /api/v1/catalog-agent/catalogs/import`). DCAT-AP is checked with **SEMIC's official
  shapes** (DCAT-AP 3.0.1, grouped as the Interoperable Europe Testbed does); DCAT 3, for which W3C
  publishes no shapes, with shapes made from the doctor's DCAT 3 rules.

The sample documents in `data/dcat-ap/` are real DCAT-AP catalogs of the EDACCIT deployment
(Puertos del Estado, ADIF, AEMET, Copernicus…).

Steps:
1. Configuration and helpers; the Provider's main catalog.
2. The catalog in each profile: a cut preview and its counts.
3. Downloads in every RDF syntax.
4. Validate a DCAT-AP document (dry run) and read the SHACL report.
5. What a broken document gets: violations and a 422.
6. Import it, see it projected back in DCAT-AP and ask the doctor.
7. Import again: updates, and `mode=create` refusing.
8. DCAT 3 in Turtle, with `http` IRIs, into the main catalog.
9. The shape groups that need background knowledge.
10. Every EDACCIT document at once, and cleaning up.

Only the Provider needs to be running; no onboarding is required for the management API.

## 1. Configuration and helpers

`show(response)` prints an HTTP answer and fails the cell on an unexpected status, so a broken step
stops the notebook. `shacl_summary(report)` prints an import report as a short table.

In [ ]:
import json
import os
from pathlib import Path

import requests

data_space_provider = os.environ.get("PROVIDER_URL", "http://127.0.0.1:1200")
catalog_api = f"{data_space_provider}/api/v1/catalog-agent/catalogs"
samples = Path("data/dcat-ap")
downloads = Path("data/out")
downloads.mkdir(parents=True, exist_ok=True)


def show(response, ok=True, limit=1500):
    """Prints an HTTP answer (cut to `limit` characters); with ok, raises unless it is 2xx."""
    try:
        text = json.dumps(response.json(), indent=2, ensure_ascii=False)
    except ValueError:
        text = response.text
    print(f"HTTP {response.status_code} {response.request.method} {response.url}")
    print(text if len(text) <= limit else text[:limit] + f"\n… ({len(text)} characters)")
    if ok:
        response.raise_for_status()
    return response


def shacl_summary(report):
    """The verdict of an import, its SHACL results and the problems of its plan."""
    validation = report["validation"]
    print(
        f"profile={report['profile']} conforms={report['conforms']} imported={report['imported']} "
        f"created={report['created']} updated={report['updated']}"
    )
    print(f"SHACL checks {validation['checks']}: {validation['violations']} violations, {validation['warnings']} warnings")
    for result in validation["results"]:
        print(f"  {result['severity']:9} [{result['check']}] {result['focusNode']}: {result['message']}")
    for issue in report["issues"]:
        print(f"  {issue['severity']:9} [plan] {issue['resource']}: {issue['message']}")

In [ ]:
main_catalog = show(requests.get(f"{catalog_api}/main"), limit=600).json()
main_catalog_id = main_catalog["@id"]

## 2. The catalog in each profile (`/catalogs/{id}/profiles/{profile}`)

`limit` keeps the first datasets and series (oldest first) with their distributions, records and
offers, so a big catalog stays light to read. `X-Total-Count` says how many there are and
`X-Shown-Count` how many the preview shows; `Link: <…>; rel="profile"` names the profile. The view is
the owner's: unlike the public routes it includes private resources, and it is the only way to read
the DSP projection without being a peer.

In [ ]:
for profile in ["dsp", "dcat3", "dcat-ap"]:
    response = requests.get(f"{catalog_api}/{main_catalog_id}/profiles/{profile}", params={"limit": 2})
    response.raise_for_status()
    print(
        f"{profile:8} {response.headers['Content-Type']:20} "
        f"shows {response.headers['X-Shown-Count']} of {response.headers['X-Total-Count']} datasets, "
        f"{len(response.content)} bytes, {response.headers['Link']}"
    )

In [ ]:
# The DCAT-AP preview: one `@graph`, the catalog first.
preview = show(requests.get(f"{catalog_api}/{main_catalog_id}/profiles/dcat-ap", params={"limit": 1}), limit=2500)

## 3. Downloads (`?download=true`, `?format=`)

`download=true` answers the whole catalog (`limit` is ignored) as an attachment named after the
catalog and the profile. `format` picks the syntax (`jsonld`, `turtle`, `ntriples`, `rdfxml`);
without it the `Accept` header decides, JSON-LD by default.

In [ ]:
for profile, format in [("dcat-ap", "turtle"), ("dcat3", "rdfxml"), ("dsp", "jsonld"), ("dcat-ap", "ntriples")]:
    response = requests.get(
        f"{catalog_api}/{main_catalog_id}/profiles/{profile}", params={"download": "true", "format": format}
    )
    response.raise_for_status()
    name = response.headers["Content-Disposition"].split('filename="')[1].rstrip('"')
    (downloads / name).write_bytes(response.content)
    print(f"{downloads / name}: {len(response.content)} bytes")

print()
print("\n".join((downloads / name).read_text().splitlines()[:5]))

In [ ]:
# The same through content negotiation.
response = requests.get(f"{catalog_api}/{main_catalog_id}/profiles/dcat-ap", headers={"Accept": "text/turtle"}, params={"limit": 1})
print(response.headers["Content-Type"])
print("\n".join(response.text.splitlines()[:25]))

## 4. Validate a DCAT-AP document (`POST /catalogs/import?dryRun=true`)

The body goes as it is, in the syntax `Content-Type` names (`application/ld+json`, `text/turtle`,
`application/rdf+xml`, `application/n-triples`). A dry run validates and plans the document without
writing anything:

- **SHACL**: DCAT-AP runs SEMIC's `base` (`shapes.ttl`: existence, literal ranges, cardinalities),
  `recommended` and `deprecated` groups by default. Violations keep the document out; warnings
  do not.
- **Plan**: catalogs, datasets, series, distributions, data services, agents and records become
  resources; any other node (a vCard contact, a period, a location) stays inside the resource using
  it. Parents come from `dcat:dataset`, `dcat:service`, `dcat:record` and `dcat:distribution`, and
  every `urn:` link must reach a resource of the document or of the catalog.

In [ ]:
palma = (samples / "fc-lastcycle-daily-palma.jsonld").read_bytes()
response = requests.post(
    f"{catalog_api}/import",
    params={"profile": "dcat-ap", "dryRun": "true"},
    headers={"Content-Type": "application/ld+json"},
    data=palma,
)
report = response.json()
shacl_summary(report)
for resource in report["resources"]:
    print(f"  {resource['action']:6} {resource['type']:14} {resource['id']}  (parent {resource.get('parent', '-')})")

## 5. A broken document: violations and a 422

Without the dataset's description or the catalog's title, SEMIC's Base shapes fail. The real import
then answers `422 Unprocessable Entity` with the same report, and nothing is written.

In [ ]:
broken = json.loads(palma)
for node in broken["@graph"]:
    if node["@type"] == "dcat:Dataset":
        node.pop("dct:description")
    if node["@type"] == "dcat:Catalog":
        node.pop("dct:title")

response = requests.post(f"{catalog_api}/import", params={"profile": "dcat-ap"}, json=broken)
print(f"HTTP {response.status_code}")
shacl_summary(response.json())

## 6. Import it, project it back and ask the doctor

The import writes every resource and its links in one transaction. The document's catalog becomes a
catalog of its own (the main catalog lists it as a sub-catalog in DSP); `dct:issued` and
`dct:modified` keep the document's dates. Projected back in DCAT-AP it passes the same shapes, and
the doctor of DCAT-AP checks with them too.

In [ ]:
response = requests.post(f"{catalog_api}/import", headers={"Content-Type": "application/ld+json"}, data=palma)
report = response.json()
shacl_summary(report)
imported_catalog_id = next(r["id"] for r in report["resources"] if r["type"] == "Catalog")

In [ ]:
show(requests.get(f"{catalog_api}/{imported_catalog_id}/profiles/dcat-ap"), limit=2000)

In [ ]:
doctor = requests.get(f"{catalog_api}/{imported_catalog_id}/doctor", params={"deep": "true"}).json()
for profile in doctor["profiles"]:
    print(f"{profile['profile']:8} compliant={profile['compliant']} violations={profile['violations']} warnings={profile['warnings']}")
    for finding in profile["findings"][:4]:
        print(f"   {finding['severity']:9} {finding['resource']}: {finding['message']}")

DSP finds what DCAT-AP does not ask for: an imported dataset has no ODRL offer, so a peer could see
it but not negotiate it until an offer is attached (`/api/v1/catalog-agent/odrl-policies`).

## 7. Import again: updates, and `mode=create`

Resources are matched by id, so importing the same document again (`mode=upsert`, the default)
replaces their descriptions one version on. With `mode=create` a document whose resources are
already stored is refused.

In [ ]:
again = requests.post(f"{catalog_api}/import", json=json.loads(palma)).json()
print(f"created={again['created']} updated={again['updated']}")

refused = requests.post(f"{catalog_api}/import", params={"mode": "create"}, json=json.loads(palma))
print(f"HTTP {refused.status_code}")
for issue in refused.json()["issues"][:3]:
    print(f"  {issue['resource']}: {issue['message']}")

## 8. DCAT 3 in Turtle, with `http` IRIs, into the main catalog

The catalog's resources are named by URNs. A resource named by any other IRI gets
`urn:<kind>:<uuid>`, with a UUIDv5 of the IRI, so the same IRI gives the same URN on every import;
the IRI is kept as `dct:identifier`. `target` sends the document's resources into an existing
catalog instead of storing its catalogs.

In [ ]:
turtle = """
@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix dct: <http://purl.org/dc/terms/> .

<https://data.example.org/catalog> a dcat:Catalog ;
    dct:title "Example open data"@en ;
    dcat:dataset <https://data.example.org/dataset/air-quality> .

<https://data.example.org/dataset/air-quality> a dcat:Dataset ;
    dct:title "Air quality, hourly"@en ;
    dct:description "Hourly NO2 and PM10 readings."@en ;
    dcat:distribution <https://data.example.org/dataset/air-quality/csv> .

<https://data.example.org/dataset/air-quality/csv> a dcat:Distribution ;
    dcat:accessURL <https://data.example.org/files/air-quality.csv> ;
    dcat:mediaType <https://www.iana.org/assignments/media-types/text/csv> .
"""
response = requests.post(
    f"{catalog_api}/import",
    params={"profile": "dcat3", "target": main_catalog_id},
    headers={"Content-Type": "text/turtle"},
    data=turtle.encode(),
)
report = response.json()
shacl_summary(report)
for resource in report["resources"]:
    print(f"  {resource['type']:12} {resource['source']} -> {resource['id']}")

In [ ]:
air_quality = next(r["id"] for r in report["resources"] if r["type"] == "Dataset")
dataset = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/datasets/{air_quality}"), limit=1200).json()
print("identifier:", dataset.get("identifier"), "in catalog:", dataset.get("inCatalog"))

## 9. Shape groups that need background knowledge (`checks=`)

SEMIC's `range` shapes check the class of every object (`dct:license` must be a
`dct:LicenseDocument`…) and `codelists` checks the EU controlled vocabularies. Both need those
objects typed, which the EU code lists do and documents seldom repeat, so they only run when asked
for. Run without the code lists, as the Testbed's "Zero" profiles do, they fail almost any real
document.

In [ ]:
response = requests.post(
    f"{catalog_api}/import",
    params={"dryRun": "true", "checks": "base,range"},
    json=json.loads(palma),
)
report = response.json()
validation = report["validation"]
print(f"checks {validation['checks']}: {validation['violations']} violations")
for result in validation["results"][:6]:
    print(f"  {result['constraint']:6} {result['message']}")

## 10. Every EDACCIT document at once, and cleaning up

Each document is validated and imported on its own; the summary shows what each one created.
Deleting a catalog removes what hangs from it (datasets, distributions, records); agents stay.

In [ ]:
imported = {}
for path in sorted(samples.glob("*.jsonld")):
    response = requests.post(f"{catalog_api}/import", headers={"Content-Type": "application/ld+json"}, data=path.read_bytes())
    report = response.json()
    imported[path.name] = next((r["id"] for r in report["resources"] if r["type"] == "Catalog"), None)
    print(
        f"{response.status_code} {path.name:52} conforms={report['conforms']!s:5} "
        f"created={report['created']:2} updated={report['updated']:2} warnings={report['validation']['warnings']}"
    )

In [ ]:
total = requests.get(f"{catalog_api}/{main_catalog_id}/profiles/dcat-ap", params={"limit": 0})
print(f"The main catalog now has {total.headers['X-Total-Count']} datasets")

In [ ]:
# Optional: remove the imported catalogs with everything hanging from them.
CLEAN_UP = False
if CLEAN_UP:
    for name, catalog_id in imported.items():
        if catalog_id:
            print(name, requests.delete(f"{catalog_api}/{catalog_id}").status_code)